# Triple-Signal Beam Search + Dual PF + Spatial ANCC + LightGBM

**Signals:**
1. TVT Particle Filter (Z-velocity)
2. ANCC Particle Filter
3. Beam Search (conservative + loose)
4. Spatial ANCC (centroid plane-fit KNN)
5. Dense ANCC (multi-point IDW)

All signals → LightGBM ensemble → TVT prediction

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.interpolate import interp1d
from scipy.spatial import cKDTree
from lightgbm import LGBMRegressor
import glob
import time
import gc
import os

# ── Data paths ──
if os.path.exists('/kaggle/input'):
    c1 = Path('/kaggle/input/rogii-wellbore-geology-prediction')
    c2 = Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction')
    if (c1 / 'test').exists():
        DATA_DIR = c1
    elif (c2 / 'test').exists():
        DATA_DIR = c2
    else:
        DATA_DIR = c1
    OUT_DIR = Path('/kaggle/working')
else:
    DATA_DIR = Path('../../data').resolve()
    OUT_DIR = Path('.')

TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
print(f'DATA_DIR: {DATA_DIR}')
print(f'Train wells: {len(list(TRAIN_DIR.glob("*__horizontal_well.csv")))}')
print(f'Test wells: {len(list(TEST_DIR.glob("*__horizontal_well.csv")))}')

DATA_DIR: /kaggle/input/competitions/rogii-wellbore-geology-prediction
Train wells: 773
Test wells: 3


In [2]:
# ── Config ──
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

LGBM_PARAMS = dict(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=64,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5.0,
    reg_alpha=0.1,
    objective='regression',
    random_state=RANDOM_STATE,
    verbose=-1,
    n_jobs=-1,
)

# TVT Particle Filter
PF_N_PARTICLES = 500
PF_MOMENTUM_ALPHA = 0.993
PF_Z_SIGMA_FLOOR = 0.005
PF_Z_SIGMA_SCALE = 2.0
PF_VELOCITY_NOISE_STD = 0.005
PF_POSITION_NOISE_STD = 0.01
PF_INIT_VELOCITY_STD = 0.02
PF_GR_SIGMA_MIN = 10.0
PF_GR_SIGMA_MAX = 60.0
PF_GR_SIGMA_DEFAULT = 30.0
PF_INIT_SPREAD_STD = 0.5
PF_RESAMPLE_THRESHOLD = 0.5
PF_ROUGHENING_STD_POS = 0.2
PF_ROUGHENING_STD_VEL = 0.003
PF_GR_ROLLING_WINDOW = 5
PF_GR_ROLLING_WEIGHT = 0.3

# ANCC Particle Filter
ANCC_ALPHA = 0.998
ANCC_RATE_NOISE_STD = 0.002
ANCC_POS_NOISE_STD = 0.005
ANCC_INIT_RATE_STD = 0.01
ANCC_INIT_SPREAD_STD = 0.3
ANCC_ROUGHENING_STD_POS = 0.1
ANCC_ROUGHENING_STD_RATE = 0.001
ANCC_N_PARTICLES = 500

# Spatial ANCC
SPATIAL_PLANE_K = 10
DENSE_SAMPLES_PER_WELL = 40
DENSE_K = 20

In [3]:
# ── Helpers ──
def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2)))


def safe_interp(x, xp, fp):
    xp = np.asarray(xp, dtype=np.float64)
    fp = np.asarray(fp, dtype=np.float64)
    mask = np.isfinite(xp) & np.isfinite(fp)
    if mask.sum() < 2:
        return np.full(len(np.asarray(x)), np.nan)
    order = np.argsort(xp[mask])
    return np.interp(np.asarray(x, float), xp[mask][order], fp[mask][order],
                     left=np.nan, right=np.nan)


def robust_slope(x, y, default=0.0):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 2:
        return default
    x, y = x[mask], y[mask]
    if np.std(x) < 1e-6:
        return default
    return float(np.polyfit(x, y, 1)[0])

In [4]:
# ── Beam Search ──
def nearest_idx(sorted_arr, value):
    idx = int(np.searchsorted(sorted_arr, value, side='left'))
    if idx >= len(sorted_arr):
        return len(sorted_arr) - 1
    if idx > 0 and abs(sorted_arr[idx - 1] - value) <= abs(sorted_arr[idx] - value):
        return idx - 1
    return idx


def smooth_gr(values, fallback, radius):
    s = pd.Series(values, dtype='float32').interpolate(
        limit_direction='both').fillna(fallback)
    if radius > 0:
        s = s.rolling(radius * 2 + 1, center=True, min_periods=1).mean()
    return s.to_numpy(dtype=np.float32)


def beam_search(gr_hidden, tw_tvt, tw_gr, start_tvt,
                beam_size=10, move_cost=20.0, emit_scale=144.0, radius=2):
    tw_tvt = np.asarray(tw_tvt, dtype=np.float32)
    tw_gr = np.asarray(tw_gr, dtype=np.float32)
    T = len(tw_tvt)
    fallback = float(np.nanmean(tw_gr))
    gr_smooth = smooth_gr(gr_hidden, fallback, radius)
    start = nearest_idx(tw_tvt, start_tvt)
    beam_idx = np.full(beam_size, start, dtype=np.int32)
    beam_cost = np.zeros(beam_size, dtype=np.float64)
    n_steps = len(gr_smooth)
    bp_state = np.empty((n_steps, beam_size), dtype=np.int32)
    bp_idx = np.empty((n_steps, beam_size), dtype=np.int32)
    deltas = np.array([-1, 0, 1], dtype=np.int32)
    for step, gr_val in enumerate(gr_smooth):
        cand_idx = np.clip(beam_idx[:, None] + deltas[None, :], 0, T - 1)
        emit = (gr_val - tw_gr[cand_idx]) ** 2 / emit_scale
        move = move_cost * np.abs(deltas)[None, :]
        cand_cost = beam_cost[:, None] + emit + move
        flat_idx = cand_idx.ravel()
        flat_cost = cand_cost.ravel()
        flat_par = np.repeat(np.arange(beam_size), 3)
        order = np.argsort(flat_cost, kind='stable')
        kept, seen = [], set()
        for o in order:
            ti = int(flat_idx[o])
            if ti not in seen:
                seen.add(ti)
                kept.append(o)
                if len(kept) == beam_size:
                    break
        while len(kept) < beam_size:
            kept.append(kept[-1])
        kept = np.array(kept, dtype=np.int32)
        bp_state[step] = flat_par[kept]
        bp_idx[step] = flat_idx[kept]
        beam_idx = flat_idx[kept].astype(np.int32)
        beam_cost = flat_cost[kept]
    path = np.empty(n_steps, dtype=np.int32)
    cur_b = int(np.argmin(beam_cost))
    for step in range(n_steps - 1, -1, -1):
        path[step] = bp_idx[step, cur_b]
        cur_b = bp_state[step, cur_b]
    return tw_tvt[path]


def run_beam(hw, tw, beam_size=10, move_cost=20.0, emit_scale=144.0, radius=3):
    """Run beam search, return (pred_tvt, std) for eval zone."""
    tw_tvt = tw['TVT'].values.astype(np.float32)
    tw_gr = tw['GR'].values.astype(np.float32)
    known = hw[hw['TVT_input'].notna()]
    evalz = hw[hw['TVT_input'].isna()]
    if len(evalz) == 0 or len(known) == 0:
        return np.array([]), np.array([])
    last_tvt = float(known['TVT_input'].iloc[-1])
    gr_filled = hw['GR'].astype(float).interpolate(limit_direction='both')
    gr_filled = gr_filled.fillna(float(np.nanmean(tw_gr)))
    hidden_gr = gr_filled.iloc[evalz.index[0]:].values.astype(np.float32)
    pred = beam_search(hidden_gr, tw_tvt, tw_gr, last_tvt,
                       beam_size=beam_size, move_cost=move_cost,
                       emit_scale=emit_scale, radius=radius)
    n_eval = len(evalz)
    pred = pred[:n_eval]
    return pred, np.zeros(n_eval, dtype=np.float32)

In [5]:
# ── TVT Particle Filter (Z-velocity signal) ──
def pf_calibrate_gr_sigma(hw, tw_tvt, tw_gr):
    known = hw[hw['TVT_input'].notna()]
    known_gr = known[known['GR'].notna()]
    if len(known_gr) < 20:
        return PF_GR_SIGMA_DEFAULT
    tw_func = interp1d(tw_tvt, tw_gr, bounds_error=False,
                       fill_value=(tw_gr[0], tw_gr[-1]))
    expected = tw_func(known_gr['TVT_input'].values)
    residuals = known_gr['GR'].values - expected
    return np.clip(np.std(residuals), PF_GR_SIGMA_MIN, PF_GR_SIGMA_MAX)


def pf_estimate_init_velocity(hw):
    known = hw[hw['TVT_input'].notna()]
    if len(known) < 10:
        return 0.0
    tail = known.tail(20)
    if len(tail) < 5:
        return 0.0
    dtvt = np.diff(tail['TVT_input'].values)
    dmd = np.diff(tail['MD'].values)
    mask = dmd > 0
    if mask.sum() < 3:
        return 0.0
    return np.median(dtvt[mask] / dmd[mask])


def pf_learn_z_beta(hw):
    known = hw[hw['TVT_input'].notna()]
    if len(known) < 30:
        return -1.0, 0.0, 0.1
    dz = np.diff(known['Z'].values)
    dtvt = np.diff(known['TVT_input'].values)
    dmd = np.diff(known['MD'].values)
    mask = dmd > 0
    if mask.sum() < 10:
        return -1.0, 0.0, 0.1
    vz = dz[mask] / dmd[mask]
    vt = dtvt[mask] / dmd[mask]
    A = np.column_stack([vz, np.ones_like(vz)])
    coef, _, _, _ = np.linalg.lstsq(A, vt, rcond=None)
    residuals = vt - (coef[0] * vz + coef[1])
    sigma = max(np.std(residuals), 0.001)
    return coef[0], coef[1], sigma


def run_pf_z_velocity(hw, tw_tvt, tw_gr, n_particles=PF_N_PARTICLES):
    tw_func_point = interp1d(tw_tvt, tw_gr, bounds_error=False,
                             fill_value=(tw_gr[0], tw_gr[-1]))
    tw_smooth_gr = pd.Series(tw_gr).rolling(
        PF_GR_ROLLING_WINDOW, center=True, min_periods=1).mean().values
    tw_func_smooth = interp1d(tw_tvt, tw_smooth_gr, bounds_error=False,
                              fill_value=(tw_smooth_gr[0], tw_smooth_gr[-1]))
    tvt_min, tvt_max = tw_tvt.min(), tw_tvt.max()
    gr_sigma = pf_calibrate_gr_sigma(hw, tw_tvt, tw_gr)
    beta, intercept, z_sigma = pf_learn_z_beta(hw)
    known = hw[hw['TVT_input'].notna()]
    evalz = hw[hw['TVT_input'].isna()]
    if len(evalz) == 0:
        return np.array([]), np.array([])
    hw_gr_smooth = hw['GR'].rolling(
        PF_GR_ROLLING_WINDOW, center=True, min_periods=1).mean()
    last_tvt = known['TVT_input'].iloc[-1]
    positions = last_tvt + np.random.normal(0, PF_INIT_SPREAD_STD, n_particles)
    init_v = pf_estimate_init_velocity(hw)
    velocities = init_v + np.random.normal(0, PF_INIT_VELOCITY_STD, n_particles)
    weights = np.ones(n_particles) / n_particles
    eval_indices = evalz.index.tolist()
    md_vals = evalz['MD'].values
    gr_vals = evalz['GR'].values
    z_vals = evalz['Z'].values
    prev_md = known['MD'].iloc[-1]
    prev_z = known['Z'].iloc[-1]
    pred_tvts = np.empty(len(evalz))
    pred_stds = np.empty(len(evalz))
    for i, idx in enumerate(eval_indices):
        d_md = md_vals[i] - prev_md
        if d_md <= 0:
            d_md = 1.0
        dz_dmd = (z_vals[i] - prev_z) / d_md
        v_expected = beta * dz_dmd + intercept
        velocities = (PF_MOMENTUM_ALPHA * velocities
                      + np.random.normal(0, PF_VELOCITY_NOISE_STD, n_particles))
        positions = (positions + velocities * d_md
                     + np.random.normal(0, PF_POSITION_NOISE_STD, n_particles))
        positions = np.clip(positions, tvt_min - 50, tvt_max + 50)
        if not np.isnan(gr_vals[i]):
            gr_smooth = hw_gr_smooth.iloc[hw.index.get_loc(idx)]
            expected_point = tw_func_point(positions)
            diff_point = gr_vals[i] - expected_point
            lik_point = np.exp(-0.5 * (diff_point / gr_sigma) ** 2)
            if not np.isnan(gr_smooth):
                expected_smooth = tw_func_smooth(positions)
                diff_smooth = gr_smooth - expected_smooth
                lik_smooth = np.exp(-0.5 * (diff_smooth / (gr_sigma * 1.5)) ** 2)
                likelihood = ((1 - PF_GR_ROLLING_WEIGHT) * lik_point
                              + PF_GR_ROLLING_WEIGHT * lik_smooth)
            else:
                likelihood = lik_point
            likelihood = np.maximum(likelihood, 1e-300)
            weights = weights * likelihood
            w_sum = weights.sum()
            if w_sum > 0:
                weights /= w_sum
            else:
                weights[:] = 1.0 / n_particles
        z_sig = max(z_sigma * PF_Z_SIGMA_SCALE, PF_Z_SIGMA_FLOOR)
        diff_v = velocities - v_expected
        lik_z = np.exp(-0.5 * (diff_v / z_sig) ** 2)
        lik_z = np.maximum(lik_z, 1e-300)
        weights = weights * lik_z
        w_sum = weights.sum()
        if w_sum > 0:
            weights /= w_sum
        else:
            weights[:] = 1.0 / n_particles
        n_eff = 1.0 / np.sum(weights ** 2)
        if n_eff < PF_RESAMPLE_THRESHOLD * n_particles:
            cum = np.cumsum(weights)
            pos_resample = (np.arange(n_particles) + np.random.uniform()) / n_particles
            indices = np.searchsorted(cum, pos_resample)
            positions = positions[indices]
            velocities = velocities[indices]
            weights[:] = 1.0 / n_particles
            positions += np.random.normal(0, PF_ROUGHENING_STD_POS, n_particles)
            velocities += np.random.normal(0, PF_ROUGHENING_STD_VEL, n_particles)
        pred_tvts[i] = np.average(positions, weights=weights)
        pred_stds[i] = np.sqrt(
            np.average((positions - pred_tvts[i]) ** 2, weights=weights))
        prev_md = md_vals[i]
        prev_z = z_vals[i]
    return pred_tvts, pred_stds

In [6]:
# ── ANCC Particle Filter (tracks S = TVT + Z) ──
def ancc_estimate_init_rate(hw):
    known = hw[hw['TVT_input'].notna()]
    if len(known) < 10:
        return 0.0
    tail = known.tail(30)
    dtvt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dmd = np.diff(tail['MD'].values)
    dancc = dtvt + dz
    mask = dmd > 0
    if mask.sum() < 3:
        return 0.0
    return np.median(dancc[mask] / dmd[mask])


def run_pf_ancc(hw, tw_tvt, tw_gr, n_particles=ANCC_N_PARTICLES):
    tvt_min, tvt_max = tw_tvt.min(), tw_tvt.max()
    gr_sigma = pf_calibrate_gr_sigma(hw, tw_tvt, tw_gr)
    known = hw[hw['TVT_input'].notna()]
    evalz = hw[hw['TVT_input'].isna()]
    if len(evalz) == 0:
        return np.array([]), np.array([])
    last_state = known['TVT_input'].iloc[-1] + known['Z'].iloc[-1]
    init_rate = ancc_estimate_init_rate(hw)
    pos = last_state + np.random.normal(0, ANCC_INIT_SPREAD_STD, n_particles)
    rate = init_rate + np.random.normal(0, ANCC_INIT_RATE_STD, n_particles)
    w = np.ones(n_particles) / n_particles
    md_vals = evalz['MD'].values
    z_vals = evalz['Z'].values
    gr_vals = evalz['GR'].values
    prev_md = known['MD'].iloc[-1]
    pred_tvts = np.empty(len(evalz))
    pred_stds = np.empty(len(evalz))
    for i in range(len(evalz)):
        d_md = md_vals[i] - prev_md
        if d_md <= 0:
            d_md = 1.0
        rate = ANCC_ALPHA * rate + np.random.normal(0, ANCC_RATE_NOISE_STD, n_particles)
        pos = pos + rate * d_md + np.random.normal(0, ANCC_POS_NOISE_STD, n_particles)
        tvt_est = pos - z_vals[i]
        tvt_clipped = np.clip(tvt_est, tvt_min - 50, tvt_max + 50)
        pos = tvt_clipped + z_vals[i]
        if not np.isnan(gr_vals[i]):
            expected_gr = np.interp(tvt_clipped, tw_tvt, tw_gr)
            diff = gr_vals[i] - expected_gr
            lik = np.exp(-0.5 * (diff / gr_sigma) ** 2)
            lik = np.maximum(lik, 1e-300)
            w *= lik
            w_sum = w.sum()
            if w_sum > 0:
                w /= w_sum
            else:
                w[:] = 1.0 / n_particles
        n_eff = 1.0 / np.sum(w ** 2)
        if n_eff < PF_RESAMPLE_THRESHOLD * n_particles:
            cum = np.cumsum(w)
            u = (np.arange(n_particles) + np.random.uniform()) / n_particles
            idx = np.searchsorted(cum, u)
            pos = pos[idx]
            rate = rate[idx]
            w[:] = 1.0 / n_particles
            pos += np.random.normal(0, ANCC_ROUGHENING_STD_POS, n_particles)
            rate += np.random.normal(0, ANCC_ROUGHENING_STD_RATE, n_particles)
        tvt_weighted = np.average(pos - z_vals[i], weights=w)
        pred_tvts[i] = tvt_weighted
        pred_stds[i] = np.sqrt(
            np.average((pos - z_vals[i] - tvt_weighted) ** 2, weights=w))
        prev_md = md_vals[i]
    return pred_tvts, pred_stds

In [7]:
# ── Spatial ANCC Imputers ──

class SpatialANCCImputer:
    """Well centroid Plane-Fit KNN imputer for ANCC."""

    def __init__(self, well_ids, data_dir):
        rows = []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            try:
                df = pd.read_csv(p, usecols=['X', 'Y', 'ANCC']).dropna()
            except Exception:
                continue
            if len(df) == 0:
                continue
            rows.append({
                'wid': wid,
                'x': float(df['X'].median()),
                'y': float(df['Y'].median()),
                'ancc_med': float(df['ANCC'].median()),
            })
        self.df = pd.DataFrame(rows)
        self.wid_to_idx = {w: i for i, w in enumerate(self.df['wid'])}
        xy = self.df[['x', 'y']].to_numpy()
        self.scale = xy.std(axis=0)
        self.scale = np.where(self.scale < 1e-3, 1.0, self.scale)
        self.tree = cKDTree(xy / self.scale)
        self.x_arr = self.df['x'].to_numpy()
        self.y_arr = self.df['y'].to_numpy()
        self.ancc_arr = self.df['ancc_med'].to_numpy()

    def impute(self, xy_query, self_wid=None, k=SPATIAL_PLANE_K):
        xy_query = np.atleast_2d(xy_query)
        q = xy_query / self.scale
        n_fetch = min(k + 5, len(self.df))
        dist, idx = self.tree.query(q, k=n_fetch)
        if self_wid is not None and self_wid in self.wid_to_idx:
            self_i = self.wid_to_idx[self_wid]
            dist = np.where(idx == self_i, np.inf, dist)
        order = np.argpartition(dist, kth=min(k - 1, n_fetch - 1), axis=1)[:, :k]
        d_k = np.take_along_axis(dist, order, axis=1)
        idx_k = np.take_along_axis(idx, order, axis=1)
        valid = np.isfinite(d_k)
        w = np.where(valid, 1.0 / (d_k + 1e-3), 0.0)
        x_n = self.x_arr[idx_k]
        y_n = self.y_arr[idx_k]
        ancc_n = self.ancc_arr[idx_k]
        wx = w * x_n
        wy = w * y_n
        ATA = np.zeros((len(xy_query), 3, 3))
        ATA[:, 0, 0] = (wx * x_n).sum(axis=1)
        ATA[:, 0, 1] = (wx * y_n).sum(axis=1)
        ATA[:, 0, 2] = wx.sum(axis=1)
        ATA[:, 1, 0] = ATA[:, 0, 1]
        ATA[:, 1, 1] = (wy * y_n).sum(axis=1)
        ATA[:, 1, 2] = wy.sum(axis=1)
        ATA[:, 2, 0] = ATA[:, 0, 2]
        ATA[:, 2, 1] = ATA[:, 1, 2]
        ATA[:, 2, 2] = w.sum(axis=1)
        ATA[:, 0, 0] += 1e-9
        ATA[:, 1, 1] += 1e-9
        ATA[:, 2, 2] += 1e-9
        ATb = np.stack([
            (wx * ancc_n).sum(axis=1),
            (wy * ancc_n).sum(axis=1),
            (w * ancc_n).sum(axis=1),
        ], axis=1)
        try:
            coef = np.linalg.solve(ATA, ATb[:, :, None]).squeeze(-1)
        except np.linalg.LinAlgError:
            sw = w.sum(axis=1)
            safe = np.where(sw < 1e-9, 1.0, sw)
            return (w * ancc_n).sum(axis=1) / safe, d_k.min(axis=1)
        ancc_pred = (coef[:, 0] * xy_query[:, 0]
                     + coef[:, 1] * xy_query[:, 1]
                     + coef[:, 2])
        d_finite = np.where(valid, d_k, np.inf)
        min_dist = d_finite.min(axis=1)
        return ancc_pred.astype(np.float32), min_dist.astype(np.float32)


class DenseANCCImputer:
    """Dense (N pts/well) IDW imputer for ANCC."""

    def __init__(self, well_ids, data_dir, samples_per_well=DENSE_SAMPLES_PER_WELL):
        xs, ys, anccs, wids = [], [], [], []
        for wid in well_ids:
            p = data_dir / f'{wid}__horizontal_well.csv'
            try:
                df = pd.read_csv(p, usecols=['X', 'Y', 'ANCC']).dropna()
            except Exception:
                continue
            if len(df) == 0:
                continue
            idx = np.linspace(0, len(df) - 1, min(samples_per_well, len(df)), dtype=int)
            sampled = df.iloc[idx]
            xs.append(sampled['X'].values)
            ys.append(sampled['Y'].values)
            anccs.append(sampled['ANCC'].values)
            wids.extend([wid] * len(sampled))
        self.xy = np.column_stack([np.concatenate(xs), np.concatenate(ys)])
        self.ancc = np.concatenate(anccs).astype(np.float32)
        self.wids = np.array(wids)
        self.wid_set = set(wids)
        self.scale = self.xy.std(axis=0)
        self.scale = np.where(self.scale < 1e-3, 1.0, self.scale)
        self.tree = cKDTree(self.xy / self.scale)

    def impute(self, xy_query, self_wid=None, k=DENSE_K):
        xy_query = np.atleast_2d(xy_query)
        q = xy_query / self.scale
        n_fetch = min(k + (DENSE_SAMPLES_PER_WELL + 5), len(self.ancc))
        dist, idx = self.tree.query(q, k=n_fetch, workers=-1)
        if self_wid is not None and self_wid in self.wid_set:
            mask_self = self.wids[idx] == self_wid
            dist = np.where(mask_self, np.inf, dist)
        order = np.argpartition(dist, kth=min(k - 1, n_fetch - 1), axis=1)[:, :k]
        d_k = np.take_along_axis(dist, order, axis=1)
        idx_k = np.take_along_axis(idx, order, axis=1)
        valid = np.isfinite(d_k)
        w = np.where(valid, 1.0 / (d_k + 1e-3), 0.0)
        sw = w.sum(axis=1)
        no_n = sw < 1e-9
        safe = np.where(no_n, 1.0, sw)
        ancc_n = self.ancc[idx_k]
        ancc_pred = (ancc_n * w).sum(axis=1) / safe
        ancc_pred = np.where(no_n, float(self.ancc.mean()), ancc_pred)
        diff_sq = (ancc_n - ancc_pred[:, None]) ** 2
        var = (diff_sq * w).sum(axis=1) / safe
        neighbor_std = np.sqrt(np.maximum(var, 0.0))
        d_finite = np.where(valid, d_k, np.inf)
        min_dist = d_finite.min(axis=1)
        return (ancc_pred.astype(np.float32),
                min_dist.astype(np.float32),
                neighbor_std.astype(np.float32))

In [8]:
# ── Feature Engineering ──
def build_features(hw, tw, pf_pred, pf_std, imputer, dense_imputer, wid,
                   beam_cons=None, beam_loose=None, is_train=True):
    """Build features for one well (Modal 02 version with spatial/dense ANCC)."""
    known = hw[hw['TVT_input'].notna()]
    evalz = hw[hw['TVT_input'].isna()]
    if len(evalz) == 0 or len(known) == 0:
        return pd.DataFrame()

    last_known = known.iloc[-1]
    last_known_tvt = float(last_known['TVT_input'])
    ps_md = float(last_known['MD'])
    ps_z = float(last_known['Z'])
    ps_x = float(last_known['X'])
    ps_y = float(last_known['Y'])

    tw_tvt = tw['TVT'].values.astype(np.float64)
    tw_gr = tw['GR'].values.astype(np.float64)
    n_eval = len(evalz)

    # ── Spatial ANCC ──
    xy_eval = evalz[['X', 'Y']].to_numpy()
    xy_known = known[['X', 'Y']].to_numpy()
    spatial_ancc_eval, spatial_dist = imputer.impute(
        xy_eval, self_wid=wid if is_train else None)
    spatial_ancc_known, _ = imputer.impute(
        xy_known, self_wid=wid if is_train else None)
    c_well = np.median(known['TVT_input'].values + known['Z'].values - spatial_ancc_known)
    z_eval = evalz['Z'].values.astype(np.float32)
    spatial_tvt = (-z_eval + spatial_ancc_eval + c_well).astype(np.float32)

    # ── Dense ANCC ──
    self_wid_arg = wid if is_train else None
    dense_ancc_eval, dense_dist, dense_nb_std_eval = dense_imputer.impute(
        xy_eval, self_wid=self_wid_arg)
    dense_ancc_known, _, dense_nb_std_known = dense_imputer.impute(
        xy_known, self_wid=self_wid_arg)
    c_well_dense = np.median(
        known['TVT_input'].values + known['Z'].values - dense_ancc_known)
    dense_tvt = (-z_eval + dense_ancc_eval + c_well_dense).astype(np.float32)

    # ── Known zone Dense reliability ──
    known_state = known['TVT_input'].values + known['Z'].values
    dense_state_known = dense_ancc_known + c_well_dense
    dense_known_residual = known_state - dense_state_known
    dense_known_rmse = np.float32(np.sqrt(np.mean(dense_known_residual ** 2)))
    dense_known_bias = np.float32(np.mean(dense_known_residual))
    dense_known_max_err = np.float32(np.max(np.abs(dense_known_residual)))
    dense_known_nb_std_mean = np.float32(np.mean(dense_nb_std_known))

    # ── GR preprocessing ──
    gr_filled = hw['GR'].astype(float).interpolate(limit_direction='both')
    known_gr_mean = known['GR'].mean()
    if np.isnan(known_gr_mean):
        known_gr_mean = gr_filled.mean()

    gr_rolls = {}
    for w_size in [21, 51]:
        roll = gr_filled.rolling(w_size, center=True, min_periods=max(2, w_size // 5))
        gr_rolls[f'gr_roll{w_size}'] = roll.mean().iloc[evalz.index].values.astype(np.float32)
        gr_rolls[f'gr_std{w_size}'] = roll.std().fillna(0).iloc[evalz.index].values.astype(np.float32)

    # ── Known segment context ──
    recent = known.tail(min(200, len(known)))
    slope_all = robust_slope(known['MD'], known['TVT_input'])
    slope_recent = robust_slope(recent['MD'], recent['TVT_input'], default=slope_all)
    slope_z_recent = robust_slope(recent['Z'], recent['TVT_input'])

    # ── Typewell match quality (known zone) ──
    known_gr_valid = known[known['GR'].notna()]
    known_tw_rmse = np.nan
    if len(known_gr_valid) > 5:
        tw_gr_at_known = safe_interp(known_gr_valid['TVT_input'].values, tw_tvt, tw_gr)
        valid_mask = np.isfinite(tw_gr_at_known)
        if valid_mask.sum() > 5:
            known_tw_rmse = np.sqrt(np.mean(
                (known_gr_valid['GR'].values[valid_mask] - tw_gr_at_known[valid_mask]) ** 2))

    # ── Trajectory derivatives ──
    md_diff = hw['MD'].diff().replace(0, np.nan)
    dz_dmd = (hw['Z'].diff() / md_diff).iloc[evalz.index].values.astype(np.float32)
    dx_dmd = (hw['X'].diff() / md_diff).iloc[evalz.index].values.astype(np.float32)
    dy_dmd = (hw['Y'].diff() / md_diff).iloc[evalz.index].values.astype(np.float32)

    # ── Build DataFrame ──
    cur = pd.DataFrame({
        # PF
        'pf_pred': pf_pred.astype(np.float32),
        'pf_std': pf_std.astype(np.float32),
        'pf_delta': (pf_pred - last_known_tvt).astype(np.float32),
        'pf_std_trend': (pf_std - pf_std[0]).astype(np.float32) if len(pf_std) > 0 else 0.0,
        'pf_std_ratio': (pf_std / max(pf_std[0], 0.01)).astype(np.float32) if len(pf_std) > 0 else 1.0,

        # Spatial
        'spatial_tvt': spatial_tvt,
        'spatial_delta': (spatial_tvt - last_known_tvt).astype(np.float32),
        'spatial_ancc': spatial_ancc_eval.astype(np.float32),
        'spatial_dist': spatial_dist.astype(np.float32),
        'c_well': np.float32(c_well),

        # Dense
        'dense_tvt': dense_tvt,
        'dense_delta': (dense_tvt - last_known_tvt).astype(np.float32),
        'dense_ancc': dense_ancc_eval.astype(np.float32),
        'dense_dist': dense_dist.astype(np.float32),
        'c_well_dense': np.float32(c_well_dense),
        'dense_nb_std': dense_nb_std_eval,

        # Dense reliability
        'dense_known_rmse': dense_known_rmse,
        'dense_known_bias': dense_known_bias,
        'dense_known_max_err': dense_known_max_err,
        'dense_known_nb_std': dense_known_nb_std_mean,

        # Beam search
        'beam_cons': beam_cons.astype(np.float32) if beam_cons is not None else np.float32(np.nan),
        'beam_loose': beam_loose.astype(np.float32) if beam_loose is not None else np.float32(np.nan),
        'beam_cons_delta': ((beam_cons - last_known_tvt) if beam_cons is not None else np.float32(np.nan)),
        'beam_loose_delta': ((beam_loose - last_known_tvt) if beam_loose is not None else np.float32(np.nan)),
        'beam_gap': ((beam_loose - beam_cons) if beam_cons is not None else np.float32(np.nan)),

        # Cross comparisons
        'pf_vs_spatial': (pf_pred - spatial_tvt).astype(np.float32),
        'pf_vs_spatial_abs': np.abs(pf_pred - spatial_tvt).astype(np.float32),
        'pf_vs_dense': (pf_pred - dense_tvt).astype(np.float32),
        'spatial_vs_dense': (spatial_tvt - dense_tvt).astype(np.float32),
        'pf_vs_beam_cons': ((pf_pred - beam_cons) if beam_cons is not None else np.float32(np.nan)),
        'dense_vs_beam_cons': ((dense_tvt - beam_cons) if beam_cons is not None else np.float32(np.nan)),

        # Position
        'md_from_ps': (evalz['MD'].values - ps_md).astype(np.float32),
        'md_from_ps_sq': ((evalz['MD'].values - ps_md) ** 2 / 1000.0).astype(np.float32),
        'z_from_ps': (evalz['Z'].values - ps_z).astype(np.float32),
        'x_from_ps': (evalz['X'].values - ps_x).astype(np.float32),
        'y_from_ps': (evalz['Y'].values - ps_y).astype(np.float32),
        'xy_dist': np.sqrt((evalz['X'].values - ps_x) ** 2
                           + (evalz['Y'].values - ps_y) ** 2).astype(np.float32),
        'eval_frac': (np.arange(n_eval) / max(n_eval - 1, 1)).astype(np.float32),
        'z': z_eval,
        'dz_dmd': dz_dmd,
        'dx_dmd': dx_dmd,
        'dy_dmd': dy_dmd,

        # GR
        'gr': evalz['GR'].values.astype(np.float32),

        # Context
        'last_known_tvt': np.float32(last_known_tvt),
        'known_len': np.int32(len(known)),
        'eval_len': np.int32(n_eval),
        'eval_len_ratio': np.float32(n_eval / max(len(known), 1)),
        'slope_all': np.float32(slope_all),
        'slope_recent': np.float32(slope_recent),
        'slope_z_recent': np.float32(slope_z_recent),
        'known_tvt_range': np.float32(known['TVT_input'].max() - known['TVT_input'].min()),
        'known_tvt_std': np.float32(known['TVT_input'].std()),
        'known_gr_mean': np.float32(known_gr_mean),
        'known_gr_std': np.float32(known['GR'].std()),
        'known_tw_rmse': np.float32(known_tw_rmse),

        # Typewell context
        'tw_tvt_range': np.float32(tw_tvt.max() - tw_tvt.min()),
        'tw_gr_mean': np.float32(tw_gr.mean()),
        'tw_gr_std': np.float32(tw_gr.std()),
    })

    for k, v in gr_rolls.items():
        cur[k] = v

    if not np.isnan(known_gr_mean):
        gr_cumdev = (gr_filled - known_gr_mean).cumsum()
        cur['gr_cumdev'] = gr_cumdev.iloc[evalz.index].values.astype(np.float32)
    else:
        cur['gr_cumdev'] = np.float32(np.nan)

    cur['tw_gr_at_pf'] = safe_interp(pf_pred, tw_tvt, tw_gr).astype(np.float32)
    cur['gr_minus_tw_at_pf'] = (cur['gr'].values - cur['tw_gr_at_pf'].values).astype(np.float32)

    for offset in [-60, 60]:
        tw_gr_off = safe_interp(pf_pred + offset, tw_tvt, tw_gr)
        cur[f'gr_tw_off_{offset}'] = (cur['gr'].values - tw_gr_off).astype(np.float32)

    cur['baseline_slope_recent'] = (
        last_known_tvt + slope_recent * cur['md_from_ps'].values).astype(np.float32)
    cur['pf_minus_slope'] = (pf_pred - cur['baseline_slope_recent'].values).astype(np.float32)
    cur['spatial_minus_slope'] = (spatial_tvt - cur['baseline_slope_recent'].values).astype(np.float32)
    cur['dense_minus_slope'] = (dense_tvt - cur['baseline_slope_recent'].values).astype(np.float32)

    if is_train:
        cur['target_tvt'] = evalz['TVT'].values.astype(np.float32)
        cur['target_residual'] = (evalz['TVT'].values - last_known_tvt).astype(np.float32)

    return cur

In [9]:
# ── Build imputers (train wells only, used for both train & test) ──
t_start = time.time()

hw_files = sorted(glob.glob(str(TRAIN_DIR / '*__horizontal_well.csv')))
train_well_ids = [Path(f).name.split('__', 1)[0] for f in hw_files]
print(f'Building imputers from {len(train_well_ids)} training wells...')

imputer = SpatialANCCImputer(train_well_ids, TRAIN_DIR)
dense_imputer = DenseANCCImputer(train_well_ids, TRAIN_DIR)
print(f'  Centroid: {len(imputer.df)} wells, Dense: {len(dense_imputer.ancc)} points')
print(f'  {time.time() - t_start:.1f}s')

Building imputers from 773 training wells...
  Centroid: 766 wells, Dense: 30640 points
  28.6s


In [10]:
# ── Build train features ──
print(f'\nBuilding train features ({len(hw_files)} wells)...')
parts = []
skipped = 0

for i, hw_path in enumerate(hw_files, 1):
    well_id = Path(hw_path).name.split('__', 1)[0]
    tw_path = TRAIN_DIR / f'{well_id}__typewell.csv'
    if not tw_path.exists():
        skipped += 1
        continue

    hw = pd.read_csv(hw_path)
    tw = pd.read_csv(tw_path)

    if not {'TVT', 'GR'}.issubset(tw.columns) or len(tw) < 2:
        skipped += 1
        continue

    evalz = hw[hw['TVT_input'].isna()]
    known = hw[hw['TVT_input'].notna()]
    if len(evalz) == 0 or len(known) < 10:
        skipped += 1
        continue

    # Filter: need ground truth TVT in eval zone for training
    eval_with_gt = hw[hw['TVT_input'].isna() & hw['TVT'].notna()]
    if len(eval_with_gt) == 0:
        skipped += 1
        continue

    tw_tvt = tw['TVT'].to_numpy(dtype=np.float32)
    tw_gr = tw['GR'].to_numpy(dtype=np.float32)

    pf_pred, pf_std = run_pf_z_velocity(hw, tw_tvt, tw_gr)
    ancc_pred, ancc_std = run_pf_ancc(hw, tw_tvt, tw_gr)
    if len(pf_pred) == 0:
        skipped += 1
        continue

    # Use ANCC PF as the primary PF signal (matches Modal version)
    if len(ancc_pred) > 0 and not np.any(np.isnan(ancc_pred)):
        pf_use = ancc_pred.astype(np.float32)
        std_use = ancc_std.astype(np.float32)
    else:
        pf_use = pf_pred.astype(np.float32)
        std_use = pf_std.astype(np.float32)

    beam_cons, _ = run_beam(hw, tw, beam_size=10, move_cost=20.0,
                            emit_scale=144.0, radius=3)
    beam_loose, _ = run_beam(hw, tw, beam_size=10, move_cost=8.0,
                             emit_scale=64.0, radius=3)

    feat = build_features(hw, tw, pf_use, std_use, imputer, dense_imputer, well_id,
                          beam_cons=beam_cons, beam_loose=beam_loose, is_train=True)
    if len(feat) == 0:
        skipped += 1
        continue

    feat['well_id'] = well_id
    parts.append(feat)

    if i % 100 == 0:
        print(f'  {i}/{len(hw_files)} wells ({time.time()-t_start:.0f}s)')

print(f'  Done: {len(parts)} wells, {skipped} skipped ({time.time()-t_start:.0f}s)')

train_df = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

print(f'\ntrain_df: {train_df.shape} | wells: {train_df["well_id"].nunique()}')


Building train features (773 wells)...
  100/773 wells (305s)
  200/773 wells (579s)
  300/773 wells (853s)
  400/773 wells (1142s)
  500/773 wells (1421s)
  600/773 wells (1708s)
  700/773 wells (1987s)
  Done: 773 wells, 0 skipped (2190s)

train_df: (3783989, 74) | wells: 773


In [11]:
# ── Train LightGBM on full training data ──
EXCLUDE = {'well_id', 'target_tvt', 'target_residual'}
FEATURE_COLS = [c for c in train_df.columns if c not in EXCLUDE]
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS[:10]}...')

X_train = train_df[FEATURE_COLS].astype(np.float32)
y_train = train_df['target_residual'].astype(np.float32)

print(f'\nTraining LightGBM on {len(X_train):,} rows...')
t0 = time.time()
model = LGBMRegressor(**LGBM_PARAMS)
model.fit(X_train, y_train)
print(f'Training done in {time.time()-t0:.1f}s')

del train_df, X_train, y_train
gc.collect()

Features (71): ['pf_pred', 'pf_std', 'pf_delta', 'pf_std_trend', 'pf_std_ratio', 'spatial_tvt', 'spatial_delta', 'spatial_ancc', 'spatial_dist', 'c_well']...

Training LightGBM on 3,783,989 rows...
Training done in 1052.8s


47

In [12]:
# ── Build test features & predict ──
np.random.seed(RANDOM_STATE)
t0 = time.time()

test_files = sorted(glob.glob(str(TEST_DIR / '*__horizontal_well.csv')))
print(f'Test wells: {len(test_files)}')

rows = []
fallback_wells = []

for i, hw_path in enumerate(test_files, 1):
    well_id = Path(hw_path).name.split('__', 1)[0]
    tw_path = TEST_DIR / f'{well_id}__typewell.csv'
    h = pd.read_csv(hw_path)
    evalz = h[h['TVT_input'].isna()]
    if len(evalz) == 0:
        continue

    known = h[h['TVT_input'].notna()]
    if len(known) < 10 or not tw_path.exists():
        fallback = float(known['TVT_input'].iloc[-1]) if len(known) > 0 else 0.0
        fallback_wells.append(well_id)
        for _, row in evalz.iterrows():
            rows.append({'id': f'{well_id}_{row.name}', 'tvt': fallback})
        continue

    tw = pd.read_csv(tw_path)
    if not {'TVT', 'GR'}.issubset(tw.columns) or len(tw) < 2:
        fallback = float(known['TVT_input'].iloc[-1])
        fallback_wells.append(well_id)
        for _, row in evalz.iterrows():
            rows.append({'id': f'{well_id}_{row.name}', 'tvt': fallback})
        continue

    tw_tvt = tw['TVT'].to_numpy(dtype=np.float32)
    tw_gr = tw['GR'].to_numpy(dtype=np.float32)

    pf_pred, pf_std = run_pf_z_velocity(h, tw_tvt, tw_gr)
    ancc_pred, ancc_std = run_pf_ancc(h, tw_tvt, tw_gr)

    if len(ancc_pred) > 0 and not np.any(np.isnan(ancc_pred)):
        pf_use = ancc_pred.astype(np.float32)
        std_use = ancc_std.astype(np.float32)
    elif len(pf_pred) > 0:
        pf_use = pf_pred.astype(np.float32)
        std_use = pf_std.astype(np.float32)
    else:
        fallback = float(known['TVT_input'].iloc[-1])
        fallback_wells.append(well_id)
        for _, row in evalz.iterrows():
            rows.append({'id': f'{well_id}_{row.name}', 'tvt': fallback})
        continue

    beam_cons, _ = run_beam(h, tw, beam_size=10, move_cost=20.0,
                            emit_scale=144.0, radius=3)
    beam_loose, _ = run_beam(h, tw, beam_size=10, move_cost=8.0,
                             emit_scale=64.0, radius=3)

    feat = build_features(h, tw, pf_use, std_use, imputer, dense_imputer, well_id,
                          beam_cons=beam_cons, beam_loose=beam_loose, is_train=False)

    if len(feat) > 0:
        X_test = feat[FEATURE_COLS].astype(np.float32)
        pred_delta = model.predict(X_test).astype(np.float32)
        lkt = feat['last_known_tvt'].iloc[0]
        pred_tvt = lkt + pred_delta
        for j, (_, row) in enumerate(evalz.iterrows()):
            rows.append({'id': f'{well_id}_{row.name}', 'tvt': float(pred_tvt[j])})
    else:
        fallback = float(known['TVT_input'].iloc[-1])
        fallback_wells.append(well_id)
        for _, row in evalz.iterrows():
            rows.append({'id': f'{well_id}_{row.name}', 'tvt': fallback})

    if i % 50 == 0:
        print(f'  {i}/{len(test_files)} wells ({time.time()-t0:.0f}s)')

print(f'\nDone in {time.time()-t0:.0f}s')
if fallback_wells:
    print(f'Fallback wells: {len(fallback_wells)}')

Test wells: 3

Done in 10s


In [13]:
# ── Save submission ──
sub = pd.DataFrame(rows)
sub.to_csv(OUT_DIR / 'submission.csv', index=False)

print(f'Submission rows: {len(sub)}')
print(sub.head())

sample_path = DATA_DIR / 'sample_submission.csv'
if sample_path.exists():
    sample = pd.read_csv(sample_path)
    print(f'\nSample submission rows: {len(sample)}')
    print(f'Our submission rows:    {len(sub)}')
    print(f'IDs match: {set(sub["id"]) == set(sample["id"])}')

Submission rows: 14151
              id           tvt
0  000d7d20_1442  11747.069336
1  000d7d20_1443  11747.117188
2  000d7d20_1444  11747.117188
3  000d7d20_1445  11747.124023
4  000d7d20_1446  11747.132812

Sample submission rows: 14151
Our submission rows:    14151
IDs match: True
